In [2]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

# 1. すでに作成した4番タイプ表を読み込む
type_df = pd.read_csv("npb_2024_main_fourth_batter_types.csv")

# 2. NPB公式の2024年チーム打撃成績URL
CENTRAL_URL = "https://npb.jp/bis/2024/stats/tmb_c.html"
PACIFIC_URL = "https://npb.jp/bis/2024/stats/tmb_p.html"

def load_team_runs(url: str) -> pd.DataFrame:
    res = requests.get(url, timeout=20)
    res.raise_for_status()
    res.encoding = res.apparent_encoding

    soup = BeautifulSoup(res.text, "html.parser")
    text = soup.get_text("\n", strip=True)

    rows = []

    if "tmb_c" in url:
        # セ・リーグ
        team_pattern = r"(DeNA|巨\s*人|中\s*日|ヤクルト|阪\s*神|広\s*島)"
    else:
        # パ・リーグ
        team_pattern = r"(ソフトバンク|日本ハム|ロッテ|楽\s*天|オリックス|西\s*武)"

    # 行の並び:
    # チーム名 打率 試合 打席 打数 得点 ...
    pattern = re.compile(
        rf"{team_pattern}\s*\.?\d+\s+(\d+)\s+\d+\s+\d+\s+(\d+)"
    )

    for m in pattern.finditer(text):
        team = re.sub(r"\s+", "", m.group(1))
        games = int(m.group(2))
        runs = int(m.group(3))
        rows.append([team, games, runs])

    if not rows:
        raise ValueError(f"チーム成績を抽出できませんでした: {url}")

    df = pd.DataFrame(rows, columns=["チーム", "試合", "得点"])
    return df

# 3. セ・パを読み込んで結合
central_df = load_team_runs(CENTRAL_URL)
pacific_df = load_team_runs(PACIFIC_URL)
team_runs_df = pd.concat([central_df, pacific_df], ignore_index=True)

# 4. 球団名をあなたのCSV側に合わせる
name_map = {
    "DeNA": "DeNA",
    "ソフトバンク": "ソフトバンク",
    "日本ハム": "日本ハム",
    "オリックス": "オリックス",
    "ロッテ": "ロッテ",
    "楽天": "楽天",
    "西武": "西武",
    "巨人": "巨人",
    "阪神": "阪神",
    "広島": "広島",
    "ヤクルト": "ヤクルト",
    "中日": "中日",
}

team_runs_df["球団"] = team_runs_df["チーム"].replace(name_map)

# 5. 数値化
team_runs_df["試合"] = pd.to_numeric(team_runs_df["試合"], errors="coerce")
team_runs_df["得点"] = pd.to_numeric(team_runs_df["得点"], errors="coerce")

# 6. 1試合平均得点を作成
team_runs_df["1試合平均得点"] = team_runs_df["得点"] / team_runs_df["試合"]

# 7. 4番タイプ表と結合
merged_df = type_df.merge(
    team_runs_df[["球団", "試合", "得点", "1試合平均得点"]],
    on="球団",
    how="left"
)

# 8. 保存
merged_df.to_csv(
    "npb_2024_main_fourth_batter_types_with_team_runs.csv",
    index=False,
    encoding="utf-8-sig"
)

print("=== チーム得点データ ===")
print(team_runs_df)

print("\n=== 4番タイプ × チーム得点 ===")
print(merged_df)

print("\n=== タイプ別平均 ===")
summary = (
    merged_df.groupby("4番タイプ")[["得点", "1試合平均得点"]]
    .mean()
    .round(3)
    .reset_index()
)
print(summary)

print("\n保存完了: npb_2024_main_fourth_batter_types_with_team_runs.csv")

=== チーム得点データ ===
       チーム   試合   得点      球団   1試合平均得点
0     DeNA  143  522    DeNA  3.650350
1       巨人  143  462      巨人  3.230769
2       中日  143  373      中日  2.608392
3     ヤクルト  143  506    ヤクルト  3.538462
4       阪神  143  485      阪神  3.391608
5       広島  143  415      広島  2.902098
6   ソフトバンク  143  607  ソフトバンク  4.244755
7      ロッテ  143  493     ロッテ  3.447552
8     日本ハム  143  532    日本ハム  3.720280
9       楽天  143  492      楽天  3.440559
10   オリックス  143  402   オリックス  2.811189
11      西武  143  350      西武  2.447552

=== 4番タイプ × チーム得点 ===
        球団     選手名  4番出場回数  本塁打    OBP    SLG    OPS  打点  長打型スコア  総合型スコア  \
0     DeNA    牧 秀悟      79   23  0.346  0.491  0.837  74   0.769   0.640   
1    オリックス    森 友哉      62    9  0.368  0.415  0.783  46  -0.556   0.412   
2   ソフトバンク   山川 穂高     143   34  0.318  0.484  0.801  99   1.181  -0.159   
3     ヤクルト   村上 宗隆     131   33  0.379  0.472  0.851  86   1.219   1.292   
4      ロッテ      ソト     116   21  0.330  0.450  0.780  88   0.252  -0.201 